# 05j - Repaired representation train/development recheck

This notebook tests whether compact bounded heads can learn from the repaired 05i-c representation. It reuses the exact registered train/development support, freezes H2, and never extracts held-out inputs or targets. A robust pass requires at least two of three fixed seeds in one feature family. No rollout or full training is performed.

## 1. Coherent checkout and GPU runtime

In [ ]:
import os, subprocess, sys
from pathlib import Path
WORKSPACE = Path('/kaggle/working/hayflow_workspace')
ELM_REPO = WORKSPACE / 'elmneuron'
if not ELM_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Zagred47/giada.git', str(ELM_REPO)], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'pandas', 'pyarrow', 'pyyaml', 'matplotlib'], check=True)
sys.path.insert(0, str(ELM_REPO))
REVISION = subprocess.check_output(['git', '-C', str(ELM_REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Revision:', REVISION)

In [ ]:
import h5py, json, numpy as np, pandas as pd, pyarrow, torch, yaml
assert torch.cuda.is_available(), 'Attiva una GPU Kaggle prima di eseguire 05j.'
print({'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0), 'experiment': 'repaired representation train/development recheck'})

## 2. Immutable inputs

Sono richiesti composite targeted, base dataset e artefatti esatti 05b--05i-c. Sono accettati ZIP o directory Kaggle estratte. L'artefatto 05i-c viene verificato integralmente prima di ricostruire il normalizzatore.

In [ ]:
import shutil, zipfile
INPUT_ROOT = Path('/kaggle/input')
def extract_zip_safely(source, destination):
    source, destination = Path(source), Path(destination)
    marker = destination / '.source_size'; stamp = str(source.stat().st_size)
    if marker.is_file() and marker.read_text().strip() == stamp: return destination
    if destination.exists(): shutil.rmtree(destination)
    destination.mkdir(parents=True); root = destination.resolve()
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            assert target == root or root in target.parents, member.filename
        archive.extractall(destination)
    marker.write_text(stamp); return destination
def first_existing(candidates, message):
    found = next((Path(p).resolve() for p in candidates if Path(p).exists()), None)
    assert found is not None, message
    return found
def artifact_source(env_name, zip_name, marker, preferred_token, message):
    candidates = [Path(os.environ[env_name]).expanduser()] if os.environ.get(env_name) else []
    candidates += list(INPUT_ROOT.rglob(zip_name))
    extracted = [p.parent for p in INPUT_ROOT.rglob(marker)]
    candidates += [p for p in extracted if preferred_token in str(p).lower()]
    candidates += extracted
    return first_existing(candidates, message)
topup_candidates = [Path(os.environ['HAYFLOW_TOPUP_V3']).expanduser()] if os.environ.get('HAYFLOW_TOPUP_V3') else []
topup_candidates += list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))
topup_candidates += [p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE = first_existing(topup_candidates, 'Top-up BAP v3 non trovato.')
TOPUP_ROOT = extract_zip_safely(TOPUP_SOURCE, '/kaggle/working/hayflow05j_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates = list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'))
assert len(manifest_candidates) == 1, manifest_candidates
COMPOSITE_MANIFEST = manifest_candidates[0]
base_candidates = [Path(os.environ['HAYFLOW_BASE_DATASET']).expanduser()] if os.environ.get('HAYFLOW_BASE_DATASET') else []
base_candidates += [p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]
base_candidates += [p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE = first_existing(base_candidates, 'Dataset base targeted v1.1 non trovato.')
CHECKPOINT_05B_SOURCE = artifact_source('HAYFLOW_05B_ARTIFACT', 'hayflow_hines_canary_v2.zip', 'canary_models.pt', 'canary', 'Artefatto 05b non trovato.')
if CHECKPOINT_05B_SOURCE.name == 'checkpoints': CHECKPOINT_05B_SOURCE = CHECKPOINT_05B_SOURCE.parent
ARTIFACT_05C_SOURCE = artifact_source('HAYFLOW_05C_ARTIFACT', 'hayflow_hines_causal_isolation.zip', 'checkpoint_forensics.json', 'causal', 'Artefatto 05c non trovato.')
ARTIFACT_05D_SOURCE = artifact_source('HAYFLOW_05D_ARTIFACT', 'hayflow_hines_residual_conditioning.zip', 'free_residual_report.json', 'residual', 'Artefatto 05d non trovato.')
ARTIFACT_05E_SOURCE = artifact_source('HAYFLOW_05E_ARTIFACT', 'hayflow_hines_segment_capacity.zip', 'capacity_probe_report.json', 'capacity', 'Artefatto 05e non trovato.')
ARTIFACT_05F_SOURCE = artifact_source('HAYFLOW_05F_ARTIFACT', 'hayflow_hines_segment_micro_canary.zip', 'micro_canary_report.json', 'micro', 'Artefatto 05f non trovato.')
ARTIFACT_05G_SOURCE = artifact_source('HAYFLOW_05G_ARTIFACT', 'hayflow_hines_optimization_audit.zip', 'optimization_support.json', 'optimization', 'Artefatto 05g non trovato.')
ARTIFACT_05H_SOURCE = artifact_source('HAYFLOW_05H_ARTIFACT', 'hayflow_hines_representation_forensics.zip', 'representation_forensics_config.json', 'representation', 'Artefatto 05h non trovato.')
ARTIFACT_05I_SOURCE = artifact_source('HAYFLOW_05I_ARTIFACT', 'hayflow_hines_state_normalization_repair.zip', 'state_normalization_repair_config.json', 'state-normalization', 'Artefatto 05i non trovato.')
ARTIFACT_05IB_SOURCE = artifact_source('HAYFLOW_05IB_ARTIFACT', 'hayflow_hines_netcon_semantic_state_repair.zip', 'netcon_semantic_repair_config.json', 'netcon-semantic', 'Artefatto 05i-b non trovato.')
ARTIFACT_05IC_SOURCE = artifact_source('HAYFLOW_05IC_ARTIFACT', 'hayflow_hines_synaptic_domain_repair.zip', 'synaptic_domain_repair_config.json', 'synaptic-domain', 'Artefatto 05i-c non trovato.')
print({'manifest': str(COMPOSITE_MANIFEST), 'base': str(BASE_SOURCE), '05b': str(CHECKPOINT_05B_SOURCE), '05c': str(ARTIFACT_05C_SOURCE), '05d': str(ARTIFACT_05D_SOURCE), '05e': str(ARTIFACT_05E_SOURCE), '05f': str(ARTIFACT_05F_SOURCE), '05g': str(ARTIFACT_05G_SOURCE), '05h': str(ARTIFACT_05H_SOURCE), '05i': str(ARTIFACT_05I_SOURCE), '05i-b': str(ARTIFACT_05IB_SOURCE), '05i-c': str(ARTIFACT_05IC_SOURCE)})

## 3. Composite and cryptographic provenance preflight

In [ ]:
import time
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started, hash_last = {}, {}
def hash_progress(name, done, total):
    now = time.monotonic(); hash_started.setdefault(name, now); percent = int(100 * done / total)
    if percent >= hash_last.get(name, -5) + 5 or done == total:
        elapsed = now - hash_started[name]; rate = done / max(elapsed, 1e-9); eta = (total - done) / max(rate, 1e-9)
        print(f'[HayFlow 05j][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min', flush=True); hash_last[name] = percent
bundle = prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST, base_source=BASE_SOURCE, progress=hash_progress)
display({'valid': bundle.manifest['valid'], 'fingerprint': bundle.fingerprint, 'transition_count': bundle.transition_count, 'physical_merge': bundle.manifest['physical_merge_performed']})
assert bundle.manifest['valid'] and bundle.transition_count == 29880 and not bundle.manifest['physical_merge_performed']

In [ ]:
from src.hayflow_model import HinesCapacityConfig, HinesConditioningConfig, HinesIsolationConfig, HinesNetConSemanticRepairConfig, HinesOptimizationAuditConfig, HinesPrototypeExperimentConfig, HinesRepairedRepresentationRecheck, HinesRepairedRepresentationRecheckConfig, HinesRepresentationForensicsConfig, HinesSegmentCanaryConfig, HinesStateNormalizationRepairConfig, HinesSynapticDomainRepairConfig
base_config = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_optimization_audit.yml').read_text())
forensic_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_representation_forensics.yml').read_text())
repair_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_state_normalization_repair.yml').read_text())
netcon_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_netcon_semantic_repair.yml').read_text())
domain_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_synaptic_domain_repair.yml').read_text())
recheck_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_repaired_representation_recheck.yml').read_text())
model_config = HinesPrototypeExperimentConfig.from_mapping(base_config['model_experiment'])
isolation_config = HinesIsolationConfig.from_mapping(base_config['isolation'])
conditioning_config = HinesConditioningConfig.from_mapping(base_config['conditioning'])
capacity_config = HinesCapacityConfig.from_mapping(base_config['capacity'])
canary_config = HinesSegmentCanaryConfig.from_mapping(base_config['micro_canary'])
audit_config = HinesOptimizationAuditConfig.from_mapping(base_config['optimization_audit'])
representation_config = HinesRepresentationForensicsConfig.from_mapping(forensic_payload['representation_forensics'])
repair_config = HinesStateNormalizationRepairConfig.from_mapping(repair_payload['state_normalization_repair'])
netcon_config = HinesNetConSemanticRepairConfig.from_mapping(netcon_payload['netcon_semantic_repair'])
domain_config = HinesSynapticDomainRepairConfig.from_mapping(domain_payload['synaptic_domain_repair'])
recheck_config = HinesRepairedRepresentationRecheckConfig.from_mapping(recheck_payload['repaired_representation_recheck'])
OUTPUT_DIR = Path('/kaggle/working/artifacts/hayflow_hines_repaired_representation_recheck')
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
session = HinesRepairedRepresentationRecheck(bundle, OUTPUT_DIR, model_config, isolation_config, conditioning_config, capacity_config, canary_config, audit_config, representation_config, CHECKPOINT_05B_SOURCE, ARTIFACT_05C_SOURCE, ARTIFACT_05D_SOURCE, ARTIFACT_05E_SOURCE, ARTIFACT_05F_SOURCE, ARTIFACT_05G_SOURCE, repair_config=repair_config, artifact_05h_source=ARTIFACT_05H_SOURCE, netcon_config=netcon_config, artifact_05i_source=ARTIFACT_05I_SOURCE, domain_config=domain_config, artifact_05ib_source=ARTIFACT_05IB_SOURCE, recheck_config=recheck_config, artifact_05ic_source=ARTIFACT_05IC_SOURCE, code_revision=REVISION)
prepare_report = session.prepare_repaired_representation_recheck()
display({'revision': REVISION, '05i-c': prepare_report['artifact_05ic'], 'families': prepare_report['input_families'], 'seeds': prepare_report['seeds'], 'pair_thresholds': prepare_report['pair_thresholds']})
assert not prepare_report['heldout_inputs_extracted']
assert not prepare_report['heldout_targets_materialized']
assert not prepare_report['full_training_authorized']

## 4. Exact repaired normalizer without held-out reads

Il normalizzatore viene ricostruito dalle sole statistiche train e deve riprodurre esattamente il fingerprint 05i-c verificato.

In [ ]:
normalizer_report = session.apply_verified_synaptic_domain_normalizer()
display(normalizer_report)
assert normalizer_report['valid']
assert normalizer_report['repaired_normalizer_fingerprint'] == normalizer_report['verified_05ic_fingerprint']
assert not normalizer_report['development_values_used_to_fit']
assert not normalizer_report['heldout_inputs_extracted']
assert not normalizer_report['heldout_targets_materialized']

## 5. Frozen features on train and development only

In [ ]:
feature_report = session.prepare_train_development_features()
display({'valid': feature_report['valid'], 'train_pairs': feature_report['train_pair_count'], 'development_pairs': feature_report['development_pair_count'], 'episode_overlap': feature_report['train_development_episode_overlap'], 'roles': feature_report['roles']})
assert feature_report['valid']
assert feature_report['train_pair_count'] == 12
assert feature_report['development_pair_count'] == 1
assert not feature_report['train_development_episode_overlap']
assert not feature_report['heldout_inputs_extracted']
assert not feature_report['heldout_boundary_targets_materialized']
assert not feature_report['heldout_event_targets_materialized']

## 6. Train-only projection diagnostic

In [ ]:
projection_report = session.run_projection_forensics()
display({k: projection_report[k] for k in ['valid', 'input_surface', 'pair_metrics', 'minimum_design_rank', 'maximum_design_rank', 'segment_count_with_projection_rmse_above_1mv', 'heldout_targets_used']})
assert projection_report['valid']
assert not projection_report['heldout_targets_used']

## 7. Bounded diagnostic heads

Nove run: tre famiglie per tre seed. Ogni run stampa epoca, train RMSE, development RMSE e ETA. Il checkpoint viene scelto solo tramite development.

In [ ]:
controls_report = session.run_repaired_bounded_controls()
display(pd.DataFrame(controls_report['family_summary']))
display(pd.DataFrame([{
    'family': row['family'], 'seed': row['seed'], 'best_epoch': row['best_epoch'],
    'train_rmse_mv': row['train']['aggregate_voltage_rmse_mv'],
    'development_rmse_mv': row['development']['aggregate_voltage_rmse_mv'],
    'train_passed': row['train_passed'], 'development_passed': row['development_passed']
} for row in controls_report['runs']]))
assert controls_report['valid']
assert not controls_report['heldout_inputs_extracted']
assert not controls_report['heldout_frozen_h2_feature_extraction_performed']
assert not controls_report['heldout_candidate_head_inference_performed']

## 8. Robust scoped decision

In [ ]:
final_report = session.finalize_repaired_representation_recheck(feature_report, projection_report, controls_report)
display({'valid': final_report['valid'], 'diagnosis': final_report['diagnosis'], 'representation_recheck_passed': final_report['representation_recheck_passed'], 'family_gate': final_report['family_gate'], 'heldout_contract': final_report['heldout_contract'], 'next_step': final_report['next_step']})
assert final_report['valid']
assert not final_report['full_training_authorized']
assert not final_report['heldout_contract']['inputs_extracted']
assert not final_report['heldout_contract']['boundary_targets_materialized']
assert not final_report['heldout_contract']['event_targets_materialized']
assert not final_report['heldout_contract']['candidate_head_inference_performed']
assert not final_report['methodology']['rollout_performed']

## 9. Create and download the diagnostic ZIP

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript, display
zip_base = Path('/kaggle/working/hayflow_hines_repaired_representation_recheck')
zip_path = Path(make_archive(str(zip_base), 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
payload = base64.b64encode(zip_path.read_bytes()).decode('ascii'); filename = zip_path.name
display(Javascript(f"""
const binary = atob('{payload}');
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const anchor = document.createElement('a');
anchor.href = url; anchor.download = '{filename}';
document.body.appendChild(anchor); anchor.click(); anchor.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
"""))
print({'zip': str(zip_path), 'size_mib': round(zip_path.stat().st_size / 2**20, 2), 'download': 'avviato dal browser'})